# 03 — Embedding index build (Colab T4)

Run this notebook **on Google Colab with T4 GPU**. Embeds all processed chunks with `BAAI/bge-small-en-v1.5`, builds ChromaDB and BM25 indices, persists to Google Drive, then downloads to local `data/vector/`.

Setup:
1. Upload `filings-rag/data/processed/` to your Google Drive at `MyDrive/filings-rag/data/processed/`
2. Open this notebook in Colab → Runtime → Change runtime type → T4 GPU
3. Run all cells

In [ ]:
# Path setup: chdir to project root + add to sys.path so imports + .env paths both work.
import os, sys
from pathlib import Path

_cwd = Path.cwd()
_root = next(
    (p for p in [_cwd] + list(_cwd.parents)
     if (p / 'requirements.txt').exists() and (p / 'src').is_dir()),
    None,
)
assert _root, f'Could not find project root from {_cwd}. Open the notebook from inside filings-rag/notebooks/.'
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f'Project root: {_root}')


In [ ]:
# Install deps in Colab
!pip install -q sentence-transformers chromadb bm25s

In [ ]:
# Smart setup supports either:
#  (a) drag-dropping processed.zip into Colab's file panel (fastest)
#  (b) mounting Drive and reading from MyDrive/filings-rag/data/processed/ or processed.zip
import os, zipfile
from pathlib import Path

PROCESSED = None

# Option A: zip uploaded directly to /content/
if Path('/content/processed.zip').exists():
    print('Found /content/processed.zip - extracting...')
    with zipfile.ZipFile('/content/processed.zip') as z:
        z.extractall('/content/')
    PROCESSED = '/content/processed'
else:
    # Option B: Drive
    from google.colab import drive
    drive.mount('/content/drive')
    drive_dir = '/content/drive/MyDrive/filings-rag/data/processed'
    drive_zip = '/content/drive/MyDrive/filings-rag/data/processed.zip'
    if Path(drive_dir).exists():
        PROCESSED = drive_dir
    elif Path(drive_zip).exists():
        print(f'Extracting {drive_zip}...')
        with zipfile.ZipFile(drive_zip) as z:
            z.extractall('/content/')
        PROCESSED = '/content/processed'

assert PROCESSED, 'No processed chunks found. Upload processed.zip to Colab files panel, OR place it at MyDrive/filings-rag/data/processed.zip'

# Indices output to /content/vector and will be zipped + downloaded at the end
VECTOR = '/content/vector'
os.makedirs(VECTOR, exist_ok=True)

print(f'PROCESSED: {PROCESSED}')
print(f'VECTOR:    {VECTOR}')


In [ ]:
# Load all processed JSONL chunks
import json
from pathlib import Path

all_chunks = []
for jsonl_path in sorted(Path(PROCESSED).rglob("*.jsonl")):
    with jsonl_path.open(encoding="utf-8") as f:
        for line in f:
            all_chunks.append(json.loads(line))

print(f"Loaded {len(all_chunks)} chunks across {len(set((c['ticker'], c['year']) for c in all_chunks))} (ticker, year) pairs")

In [ ]:
# Embed with BGE-small on T4 GPU, batched
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model = SentenceTransformer("BAAI/bge-small-en-v1.5", device=device)
texts = [c["text"] for c in all_chunks]

embeddings = model.encode(
    texts,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True,
)
print(f"Shape: {embeddings.shape}")

In [ ]:
# Persist to Chroma (cosine), batched in 10-company chunks to survive any timeout
import chromadb

client = chromadb.PersistentClient(path=f"{VECTOR}/chroma")
coll = client.get_or_create_collection("filings", metadata={"hnsw:space": "cosine"})

BATCH = 1000
for i in range(0, len(all_chunks), BATCH):
    batch = all_chunks[i:i + BATCH]
    coll.add(
        documents=[c["text"] for c in batch],
        metadatas=[{"ticker": c["ticker"], "year": c["year"], "page": c["page"]} for c in batch],
        ids=[c["chunk_hash"] for c in batch],
        embeddings=embeddings[i:i + BATCH].tolist(),
    )
    print(f"  Indexed {i + len(batch)}/{len(all_chunks)}")
print("Chroma persistence complete.")

In [ ]:
# Build BM25 index over the same chunk IDs
import bm25s
import pickle

corpus_tokens = bm25s.tokenize([c["text"] for c in all_chunks], stopwords="en")
retriever = bm25s.BM25()
retriever.index(corpus_tokens)

with open(f"{VECTOR}/bm25.pkl", "wb") as f:
    pickle.dump({"retriever": retriever, "ids": [c["chunk_hash"] for c in all_chunks]}, f)
print("BM25 index saved.")

In [ ]:
# Smoke-test both indices with 3 sanity queries
sanity_qs = [
    "climate-related risks",
    "going concern statement",
    "executive remuneration policy",
]

for q in sanity_qs:
    q_vec = model.encode([q], normalize_embeddings=True).tolist()
    res = coll.query(query_embeddings=q_vec, n_results=3)
    print(f"\n--- {q!r} ---")
    for _id, meta, dist in zip(res["ids"][0], res["metadatas"][0], res["distances"][0]):
        print(f"  {meta['ticker']}|{meta['year']}|p.{meta['page']}  score={1-dist:.3f}")

## Download the indices

Run the cell below - it zips `/content/vector` and opens a browser download dialog. Save the resulting `vector.zip`, then unzip it into `filings-rag/data/vector/` locally.


In [ ]:
# Zip the built indices and trigger a browser download
import shutil
from google.colab import files

zip_path = shutil.make_archive('/content/vector', 'zip', '/content/vector')
print(f'Zipped to {zip_path}')
files.download(zip_path)
